In [23]:
import pandas as pd
import numpy as np
import joblib
import os
import time

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

RANDOM_STATE = 42
DATA_PATH    = '../data/csv/business_features.csv'
MODELS_DIR   = '../data/models/predictor'
SPLIT_DIR    = '../data/splits'

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(SPLIT_DIR, exist_ok=True)

In [24]:
df = pd.read_csv(DATA_PATH)

print(f'Shape del dataset: {df.shape}')
print(f'\nRatio desbalance: {df["is_open"].value_counts()[0] / df["is_open"].value_counts()[1]:.2f}:1')

Shape del dataset: (150243, 76)

Distribución de is_open:
is_open
1    119603
0     30640
Name: count, dtype: int64

Ratio desbalance: 0.26:1


In [26]:
X = df.drop(columns=['business_id', 'is_open'])
y = df['is_open']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)

print(f'Train: {X_train.shape[0]:,} instancias')
print(f'Test:  {X_test.shape[0]:,} instancias')
print(f'\nDistribución en train:')
print(y_train.value_counts(normalize=True).round(3))
print(f'\nDistribución en test:')
print(y_test.value_counts(normalize=True).round(3))

X_train.to_csv(f'{SPLIT_DIR}/X_train.csv', index=False)
X_test.to_csv(f'{SPLIT_DIR}/X_test.csv', index=False)
y_train.to_csv(f'{SPLIT_DIR}/y_train.csv', index=False)
y_test.to_csv(f'{SPLIT_DIR}/y_test.csv', index=False)
print(f'\nSplit guardado en {SPLIT_DIR}/')

Train: 120,194 instancias
Test:  30,049 instancias

Distribución en train:
is_open
1    0.796
0    0.204
Name: proportion, dtype: float64

Distribución en test:
is_open
1    0.796
0    0.204
Name: proportion, dtype: float64

Split guardado en ../data/splits/


In [27]:
# REGRESIÓN LOGÍSTICA - En teoría va a ser el peor modelo

pipeline_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(
        C=1.0,
        max_iter=1000,
        solver='lbfgs',
        class_weight='balanced',
        random_state=RANDOM_STATE,
    ))
])

t0 = time.time()
pipeline_lr.fit(X_train, y_train)
print(f'Completado en {time.time() - t0:.1f}s')

joblib.dump(pipeline_lr, f'{MODELS_DIR}/logistic_regression.pkl')
print(f'Modelo guardado en {MODELS_DIR}/logistic_regression.pkl')

Entrenando Regresión Logística...
Completado en 1.9s
Modelo guardado en ../data/models/predictor/logistic_regression.pkl


In [28]:
pipeline_rf = Pipeline([
    ('clf', RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        min_samples_leaf=2,
        class_weight='balanced',
        random_state=RANDOM_STATE,
    ))
])

t0 = time.time()
pipeline_rf.fit(X_train, y_train)
print(f'Completado en {time.time() - t0:.1f}s')

joblib.dump(pipeline_rf, f'{MODELS_DIR}/random_forest.pkl')
print(f'Modelo guardado en {MODELS_DIR}/random_forest.pkl')

Entrenando Random Forest...
Completado en 43.4s
Modelo guardado en ../data/models/predictor/random_forest.pkl


In [29]:
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos
print(f'scale_pos_weight: {scale_pos_weight:.4f} (neg={neg:,}, pos={pos:,})')

pipeline_xgb = Pipeline([
    ('clf', XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        eval_metric='logloss',
        verbosity=0
    ))
])

t0 = time.time()
pipeline_xgb.fit(X_train, y_train)
print(f'Completado en {time.time() - t0:.1f}s')

joblib.dump(pipeline_xgb, f'{MODELS_DIR}/xgboost.pkl')
print(f'Modelo guardado en {MODELS_DIR}/xgboost.pkl')

scale_pos_weight: 0.2562 (neg=24,512, pos=95,682)
Entrenando XGBoost...
Completado en 1.6s
Modelo guardado en ../data/models/predictor/xgboost.pkl
